In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    coalesce,
    desc,
    lit,
    lower,
    regexp_replace,
    timestamp_seconds,
    trim,
    try_to_timestamp,
    when,
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, DateType, DoubleType
from pathlib import Path

spark = SparkSession.builder \
    .appName("DataCleaningPipelineForOrdersData") \
    .getOrCreate()



In [30]:
spark


In [31]:
# 2. Xử lý Schema Inconsistency (Cat 5): Định nghĩa Schema thủ công dựa trên yêu cầu
# Việc định nghĩa rõ ràng kiểu dữ liệu ngăn chặn việc Spark đoán sai kiểu từ nguồn bẩn [3, 4]
order_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("shipping_address_id", IntegerType(), True),
    StructField("billing_address_id", IntegerType(), True),
    StructField("coupon_id", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("subtotal", DoubleType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("tax_amount", DoubleType(), True),
    StructField("shipping_cost", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("order_date", StringType(), True),  # Đọc tạm String để xử lý format
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True)
])

In [32]:
base = Path("/home/vanhcao/vietanh/graduation project")

raw_orders_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(order_schema)
    .load(str(base / "data/csv/orders.csv"))
)

raw_orders_df.show()


+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
|  id|customer_id|shipping_address_id|billing_address_id|coupon_id|   status|           subtotal|discount_amount|tax_amount|shipping_cost|total_amount|         order_date|         created_at|         updated_at|
+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
| 230|       2926|               4428|              4428|     NULL|     None|             712.69|            0.0|     57.02|          0.0|      769.71|2024-09-17 07:55:24|               NULL|2024-09-17 07:55:24|
|3248|        801|               1218|              1218|     NULL|  shipped|             992.76|           NULL|    -79.42|         NULL|     1072.18|2

In [33]:
# 4. Làm sạch, Chuẩn hóa và Ép kiểu (Transformation)
# Chuẩn hóa datetime: định dạng "yyyy-MM-dd HH:mm:ss" hoặc Unix epoch (giây) như chuỗi số (vd: 1737787974)


def parse_mixed_ts(column_name):
    x = trim(col(column_name))
    return coalesce(
        try_to_timestamp(x, lit("yyyy-MM-dd HH:mm:ss")),
        when(x.rlike(r"^\d+$"), timestamp_seconds(x.cast("long"))),
    )


cleaned_step1_df = (
    raw_orders_df.withColumn("status", trim(lower(col("status"))))
    .withColumn("order_date", parse_mixed_ts("order_date"))
    .withColumn("created_at", parse_mixed_ts("created_at"))
    .withColumn("updated_at", parse_mixed_ts("updated_at"))
)

cleaned_step1_df.show()

+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
|  id|customer_id|shipping_address_id|billing_address_id|coupon_id|   status|           subtotal|discount_amount|tax_amount|shipping_cost|total_amount|         order_date|         created_at|         updated_at|
+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
| 230|       2926|               4428|              4428|     NULL|     none|             712.69|            0.0|     57.02|          0.0|      769.71|2024-09-17 07:55:24|               NULL|2024-09-17 07:55:24|
|3248|        801|               1218|              1218|     NULL|  shipped|             992.76|           NULL|    -79.42|         NULL|     1072.18|2

In [35]:
numeric_cols = ["subtotal", "discount_amount", "tax_amount", "shipping_cost", "total_amount"]
fill_values = {**{c: 0.0 for c in numeric_cols}, "status": "unknown"}
cleaned_step2_df = cleaned_step1_df.na.fill(fill_values)
cleaned_step2_df.show()

+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
|  id|customer_id|shipping_address_id|billing_address_id|coupon_id|   status|           subtotal|discount_amount|tax_amount|shipping_cost|total_amount|         order_date|         created_at|         updated_at|
+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
| 230|       2926|               4428|              4428|     NULL|     none|             712.69|            0.0|     57.02|          0.0|      769.71|2024-09-17 07:55:24|               NULL|2024-09-17 07:55:24|
|3248|        801|               1218|              1218|     NULL|  shipped|             992.76|            0.0|    -79.42|          0.0|     1072.18|2

In [37]:
final_cleaned_df = cleaned_step2_df.dropna(subset=["id", "customer_id"])
final_cleaned_df.show()

+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
|  id|customer_id|shipping_address_id|billing_address_id|coupon_id|   status|           subtotal|discount_amount|tax_amount|shipping_cost|total_amount|         order_date|         created_at|         updated_at|
+----+-----------+-------------------+------------------+---------+---------+-------------------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
| 230|       2926|               4428|              4428|     NULL|     none|             712.69|            0.0|     57.02|          0.0|      769.71|2024-09-17 07:55:24|               NULL|2024-09-17 07:55:24|
|3248|        801|               1218|              1218|     NULL|  shipped|             992.76|            0.0|    -79.42|          0.0|     1072.18|2

In [38]:
final_orders_df = final_cleaned_df.dropDuplicates(["id"])
final_orders_df.show()

26/04/19 09:55:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-----------+-------------------+------------------+---------+----------+---------+---------------+----------+-------------+-------------------+-------------------+-------------------+-------------------+
| id|customer_id|shipping_address_id|billing_address_id|coupon_id|    status| subtotal|discount_amount|tax_amount|shipping_cost|       total_amount|         order_date|         created_at|         updated_at|
+---+-----------+-------------------+------------------+---------+----------+---------+---------------+----------+-------------+-------------------+-------------------+-------------------+-------------------+
|  1|       1120|               1683|              1683|     NULL|processing|      0.0|            0.0|    258.48|          0.0|           -3489.43|2025-05-12 01:08:25|2025-05-12 01:08:25|2025-05-12 01:08:25|
|  2|       2303|               3485|              3485|     NULL| delivered|    19.47|            0.0|       0.0|         9.99|              31.02|2025-06-06 17:36

In [40]:
final_orders_df.printSchema()
final_orders_df.show(5)

root
 |-- id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- shipping_address_id: integer (nullable = true)
 |-- billing_address_id: integer (nullable = true)
 |-- coupon_id: integer (nullable = true)
 |-- status: string (nullable = false)
 |-- subtotal: double (nullable = false)
 |-- discount_amount: double (nullable = false)
 |-- tax_amount: double (nullable = false)
 |-- shipping_cost: double (nullable = false)
 |-- total_amount: double (nullable = false)
 |-- order_date: timestamp (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+---+-----------+-------------------+------------------+---------+----------+--------+---------------+----------+-------------+------------+-------------------+-------------------+-------------------+
| id|customer_id|shipping_address_id|billing_address_id|coupon_id|    status|subtotal|discount_amount|tax_amount|shipping_cost|total_amount|         order_date|         cre